In [94]:
import pandas as pd 
import numpy as np

import matplotlib.pyplot as plt

In [95]:
df=pd.read_csv("course_lead_scoring_2026.csv")
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


Data preparation


In [96]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [97]:
df.dtypes

lead_source                  object
industry                     object
employment_status            object
location                     object
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [98]:
catg=list(df.dtypes[df.dtypes=='object'].index)
numl=list(df.dtypes[(df.dtypes=='float64') |(df.dtypes=='int64') ].index)


In [99]:
for c  in catg:
    df[c]=df[c].fillna('NA')
    
for v in numl:
    df[v]=df[v].fillna(0.0)
    

In [100]:
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [101]:
numl=['annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score']

In [102]:
from sklearn.model_selection import train_test_split

In [103]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)
len(df_train),len(df_val),len(df_test)

(3000, 1000, 1000)

In [104]:
df_train=df_train.reset_index(drop=True)
df_test=df_test.reset_index(drop=True)
df_val=df_val.reset_index(drop=True)

y_train=df_train.converted.values
y_test=df_test.converted.values
y_val=df_val.converted.values

del df_train['converted']
del df_test['converted']
del df_val['converted']

Question 1: ROC AUC feature importance


In [105]:
from sklearn.metrics import roc_auc_score

In [106]:
for v in numl:
     auc_v=roc_auc_score(y_train, df_train[v])
     if auc_v < 0.5:
        auc_v=roc_auc_score(y_train, -df_train[v]) 
     print(v,round(auc_v,3))

annual_income 0.608
number_of_courses_viewed 0.723
interaction_count 0.765
lead_score 0.788


Question 2: Training the model


In [107]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [108]:
train_dicts=df_train.to_dict(orient='records')
val_dicts=df_val.to_dict(orient='records')

dv=DictVectorizer(sparse=False)
x_train=dv.fit_transform(train_dicts)
x_val=dv.transform(val_dicts)

model=LogisticRegression(solver='liblinear',C=1.0, max_iter=1000)
model.fit(x_train,y_train)

y_pred=model.predict_proba(x_val)[:,1]
auc=roc_auc_score(y_val,y_pred)
print(round(auc,3))

0.732


Question 3: Precision and Recall

In [109]:
scores=[]
thresholds=np.linspace(0, 1, 101)
for t in thresholds:
    actual_positive=(y_val==1)
    actual_negative=(y_val==0)
    predict_positive=(y_pred>=t)
    predict_negative=(y_pred<t)

    tp=(predict_positive & actual_positive).sum()
    tn=(predict_negative & actual_negative).sum()

    fp=(predict_positive & actual_negative).sum()
    fn=(predict_negative & actual_positive).sum()
    
    scores.append((t, tp, fp, fn, tn))
    
columns = ['threshold', 'tp', 'fp', 'fn', 'tn']
df_scores = pd.DataFrame(scores, columns=columns)

df_scores['Presion'] = df_scores.tp / (df_scores.tp + df_scores.fp)
df_scores['recall'] = df_scores.tp / (df_scores.tp + df_scores.fn)

In [110]:
pd.set_option('display.max_rows', None)
df_scores

,threshold,tp,fp,fn,tn,Presion,recall
0,0.00,586,414,0,0,0.586000,1.000000
1,0.01,586,414,0,0,0.586000,1.000000
2,0.02,586,414,0,0,0.586000,1.000000
3,0.03,586,414,0,0,0.586000,1.000000
4,0.04,586,414,0,0,0.586000,1.000000
5,0.05,586,414,0,0,0.586000,1.000000
6,0.06,586,414,0,0,0.586000,1.000000
7,0.07,586,414,0,0,0.586000,1.000000
8,0.08,586,414,0,0,0.586000,1.000000
9,0.09,586,414,0,0,0.586000,1.000000


Question 4: F1 score

In [111]:
df_scores['f1_score'] = 2*((df_scores['Presion']*df_scores['recall']) / (df_scores['Presion']+df_scores['recall']))


In [112]:
df_scores.sort_values(by='f1_score',ascending=False)


,threshold,tp,fp,fn,tn,Presion,recall,f1_score
41,0.41,572,352,14,62,0.619048,0.976109,0.757616
42,0.42,568,348,18,66,0.620087,0.969283,0.756325
43,0.43,564,342,22,72,0.622517,0.962457,0.756032
45,0.45,554,328,32,86,0.628118,0.945392,0.754768
44,0.44,560,338,26,76,0.623608,0.955631,0.754717
46,0.46,548,319,38,95,0.632065,0.935154,0.754301
48,0.48,540,306,46,108,0.638298,0.921502,0.754190
40,0.40,573,362,13,52,0.612834,0.977816,0.753452
47,0.47,543,313,43,101,0.634346,0.926621,0.753121
50,0.50,531,294,55,120,0.643636,0.906143,0.752658


Question 5: 5-Fold CV

In [113]:
from sklearn.model_selection import KFold

In [ ]:
def train(df_train, y_train, C):
    dicts = df_train[catg+numl].to_dict(orient='records')

    dv = DictVectorizer(sparse=False)
    X_train=dv.fit_transform(dicts)

    model = LogisticRegression(solver='liblinear',C=C, max_iter=1000)
    model.fit(X_train, y_train)
    
    return dv, model 

In [115]:
def predict(df, dv, model):
    dicts = df[catg+numl].to_dict(orient='records')

    X = dv.transform(dicts)
    y_pred = model.predict_proba(X)[:, 1]

    return y_pred

In [118]:
scores_auc_roc=[]
kfold=KFold(n_splits=5, shuffle=True, random_state=1)
for train_idx, val_idx in kfold.split(df_full_train):
    df_train=df_full_train.iloc[train_idx]
    df_val=df_full_train.iloc[val_idx]
    
    df_train=df_train.reset_index(drop=True)
    df_val=df_val.reset_index(drop=True)

    y_train = df_train.converted.values
    y_val = df_val.converted.values

    del df_train['converted']
    del df_val['converted']
    
    dv,model=train(df_train, y_train,1.0)
    y_pred=predict(df_val,dv,model)

    scores_auc_roc.append(roc_auc_score(y_val, y_pred))

print(scores_auc_roc)
print(np.std(scores_auc_roc).round(3))

[0.7305601027122453, 0.7295726148196622, 0.7363428071259397, 0.7412454717957564, 0.7209138590326581]
0.007



Question 6: Hyperparameter Tuning


In [120]:
for C in [0.000001, 0.001, 1]:
    scores_auc_roc = []
    kfold=KFold(n_splits=5, shuffle=True, random_state=1)
    for train_idx, val_idx in kfold.split(df_full_train):
        df_train=df_full_train.iloc[train_idx]
        df_val=df_full_train.iloc[val_idx]
        
        df_train=df_train.reset_index(drop=True)
        df_val=df_val.reset_index(drop=True)

        y_train=df_train.converted.values
        y_val=df_val.converted.values

        del df_train['converted']
        del df_val['converted']
        
        dv,model=train(df_train, y_train,C)
        y_pred=predict(df_val,dv,model)

        scores_auc_roc.append(roc_auc_score(y_val, y_pred))

    print(C,round(np.mean(scores_auc_roc),3),round(np.std(scores_auc_roc),3))

1e-06 0.617 0.019
0.001 0.749 0.01
1 0.732 0.007
